# CMIP6 ACCESS-CM2 Example Download and Plot - North Carolina
This notebook downloads a single CMIP6 model (ACCESS-CM2) for North Carolina. Update parameters as needed.

## 1. Import Required Libraries
Import libraries for CDS API, file handling, and plotting.

In [ ]:
from pathlib import Path

import cdsapi
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import box

from cmip6 import (
    normalize_model_id,
    sanitize_filename,
    open_dataset_auto,
    combine_experiments,
    generate_year_range,
    compute_anomalies,
    average_tasmax_tasmin,
    cleanup_intermediate_files,
)

## 2. Define Model Parameters
Choose a model, scenario, and variable to download from CDS.

In [ ]:
EXCEL_FILE = "CMIP6_Model_Availability.xlsx"
SHEET_NAME = "Intersection_All_SSPs"

MODEL = "ACCESS-CM2"  # change to any model in the sheet
EXPERIMENTS = ["historical", "ssp126", "ssp245", "ssp585"]
VARIABLE = "tasmax"  # tasmax or tasmin

DOWNLOAD_DIR = Path("CMIP6")
LOCATION = "NorthCarolina"  # Organize downloads by location
DATASET = "projections-cmip6"
TEMPORAL_RESOLUTION = "monthly"
MONTHS = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
AREA = [36.6, -84.3, 33.8, -75.4]  # North Carolina [N, W, S, E]

VARIABLE_MAP = {
    "tasmax": "daily_maximum_near_surface_air_temperature",
    "tasmin": "daily_minimum_near_surface_air_temperature",
}

EXPERIMENT_API_MAP = {
    "historical": "historical",
    "ssp126": "ssp1_2_6",
    "ssp245": "ssp2_4_5",
    "ssp585": "ssp5_8_5",
}

EXPERIMENT_YEARS = {
    "historical": (1850, 2014),
    "ssp126": (2015, 2300),
    "ssp245": (2015, 2300),
    "ssp585": (2015, 2300),
}

### 2a. Map of North Carolina and CMIP6 Area
Overlay the CMIP6 bounding box on the North Carolina state outline.

In [ ]:
states_url = "https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_state_20m.zip"
states = gpd.read_file(states_url)
nc = states[states["NAME"] == "North Carolina"].to_crs("EPSG:4326")

north, west, south, east = AREA
bbox = box(west, south, east, north)
bbox_gdf = gpd.GeoSeries([bbox], crs="EPSG:4326")

ax = nc.plot(figsize=(8, 8), color="lightblue", edgecolor="black", linewidth=1.2, alpha=0.5)
bbox_gdf.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2.5, label="CMIP6 bounds")
ax.set_title("North Carolina with CMIP6 Bounding Box")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.show()

## 3. Download Model Artifact
Retrieve a single model and it's historical, SSP126, SSP245, and SSP585 scenarios from CDS using cdsapi.

In [ ]:
def read_availability(path: str, sheet: str):
    df = pd.read_excel(path, sheet_name=sheet)
    required = {"Model", "tasmax", "tasmin"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {', '.join(sorted(missing))}")
    return df

df_availability = read_availability(EXCEL_FILE, SHEET_NAME)

if VARIABLE not in VARIABLE_MAP:
    raise ValueError(f"Unsupported variable: {VARIABLE}")

for exp in EXPERIMENTS:
    if exp not in EXPERIMENT_API_MAP:
        raise ValueError(f"Unsupported experiment: {exp}")
    if exp not in EXPERIMENT_YEARS:
        raise ValueError(f"Missing years for experiment: {exp}")

if MODEL not in df_availability["Model"].values:
    raise ValueError(f"Model not found in sheet: {MODEL}")

availability_value = df_availability.set_index("Model")[VARIABLE].get(MODEL, "")
if str(availability_value).lower() != "available":
    raise ValueError(f"{MODEL} {VARIABLE} is not marked Available in the sheet")

model_id = normalize_model_id(MODEL)
file_model = sanitize_filename(MODEL)
variable_name = VARIABLE_MAP[VARIABLE]
model_dir = DOWNLOAD_DIR / LOCATION / file_model  # Organize by location then model
model_dir.mkdir(parents=True, exist_ok=True)

client = cdsapi.Client()
LOCAL_PATH = None

for exp in EXPERIMENTS:
    start_year, end_year = EXPERIMENT_YEARS[exp]
    api_experiment = EXPERIMENT_API_MAP[exp]

    output_name = f"cmip6_{VARIABLE}_{file_model}_{exp}_{start_year}-{end_year}.nc"
    LOCAL_PATH = model_dir / output_name

    if LOCAL_PATH.exists():
        print(f"File already exists, skipping: {LOCAL_PATH}")
        continue

    request = {
        "temporal_resolution": TEMPORAL_RESOLUTION,
        "experiment": api_experiment,
        "variable": variable_name,
        "model": model_id,
        "month": MONTHS,
        "year": generate_year_range(start_year, end_year),
        "area": AREA,
        "format": "netcdf",
    }

    result = client.retrieve(DATASET, request)

## 4. Load Model Metadata
Load metadata from the last downloaded NetCDF.

In [ ]:
if LOCAL_PATH.exists():
    ds = open_dataset_auto(LOCAL_PATH, download_dir=model_dir)
    print(ds)
    print("Variables:", list(ds.data_vars))
else:
    ds = None
    print("File not found. Download first.")

In [ ]:
print("\nDataset Time Ranges:")
print("=" * 60)

for exp in EXPERIMENTS:
    start_year, end_year = EXPERIMENT_YEARS[exp]
    output_name = f"cmip6_{VARIABLE}_{file_model}_{exp}_{start_year}-{end_year}.nc"
    exp_path = model_dir / output_name

    if exp_path.exists():
        try:
            ds_exp = open_dataset_auto(exp_path)
            time_var = ds_exp.time
            time_min = pd.Timestamp(time_var.values[0]).year
            time_max = pd.Timestamp(time_var.values[-1]).year
            n_timesteps = len(time_var)
            print(f"{exp:12s}: {time_min} - {time_max} ({n_timesteps} timesteps)")
        except Exception as e:
            print(f"{exp:12s}: Error reading time - {e}")
    else:
        print(f"{exp:12s}: File not found")

print("=" * 60)

## 5. Plot Model Summary
Plot a time series of the spatial mean for the chosen variable.

In [ ]:
color_map = {
    "historical": "black",
    "ssp126": "green",
    "ssp245": "yellow",
    "ssp585": "red",
}

fig, ax = plt.subplots(figsize=(12, 6))

for exp in EXPERIMENTS:
    start_year, end_year = EXPERIMENT_YEARS[exp]
    output_name = f"cmip6_{VARIABLE}_{file_model}_{exp}_{start_year}-{end_year}.nc"
    exp_path = model_dir / output_name

    if exp_path.exists():
        try:
            ds_exp = open_dataset_auto(exp_path)
            var_name = VARIABLE
            if var_name not in ds_exp.data_vars:
                var_name = list(ds_exp.data_vars)[0]
            da_exp = ds_exp[var_name]
            spatial_dims = [d for d in da_exp.dims if d != "time"]
            if "time" in da_exp.dims:
                series = da_exp.mean(dim=spatial_dims) if spatial_dims else da_exp
                series.plot(ax=ax, label=exp, color=color_map.get(exp, "gray"), linewidth=2)
        except Exception as e:
            print(f"Error loading {exp}: {e}")
    else:
        print(f"File not found for {exp}: {exp_path}")

ax.set_title(f"{VARIABLE} Time Series - {MODEL}", fontsize=14)
ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel(f"{VARIABLE} (K)", fontsize=12)
ax.legend(loc="best", fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5a. Combine All Experiments into Single File
Merge historical and all scenario data into one NetCDF file per model.

In [ ]:
combine_experiments(model_dir, VARIABLE, file_model, EXPERIMENT_YEARS)

## 6. Compute Anomalies and Average tasmax + tasmin

Download both tasmax and tasmin, compute anomalies relative to 1961-1990 baseline, then average them together.


In [ ]:
# Step 1: Download both tasmax and tasmin
# First, re-download with tasmin (change VARIABLE in cell 3 to "tasmin" and re-run download cell)
# Or directly load if already downloaded:

model_dir = DOWNLOAD_DIR / file_model

# Load tasmax combined file
tasmax_combined_file = model_dir / f"cmip6_tasmax_{file_model}_combined_1850-2300.nc"
if not tasmax_combined_file.exists():
    print(f"tasmax combined file not found. Download tasmax first.")
else:
    ds_tasmax = open_dataset_auto(tasmax_combined_file, download_dir=model_dir)
    print(f"Loaded tasmax: {ds_tasmax}")

# Load tasmin combined file
tasmin_combined_file = model_dir / f"cmip6_tasmin_{file_model}_combined_1850-2300.nc"
if not tasmin_combined_file.exists():
    print(f"tasmin combined file not found. Download tasmin first.")
else:
    ds_tasmin = open_dataset_auto(tasmin_combined_file, download_dir=model_dir)
    print(f"Loaded tasmin: {ds_tasmin}")


In [ ]:
# Step 2: Compute anomalies for both tasmax and tasmin
# Baseline period: 1961-1990

ds_tasmax_anom = compute_anomalies(ds_tasmax, baseline_start_year=1961, baseline_end_year=1990)
ds_tasmin_anom = compute_anomalies(ds_tasmin, baseline_start_year=1961, baseline_end_year=1990)

print("\nTaskmax anomalies:")
print(ds_tasmax_anom)
print("\nTaskmin anomalies:")
print(ds_tasmin_anom)


In [ ]:
# Step 3: Average tasmax and tasmin anomalies together
# This creates a single temperature variable that is the mean of both

ds_avg_anom = average_tasmax_tasmin(ds_tasmax_anom, ds_tasmin_anom)

print("Average temperature anomalies (tasmax + tasmin) / 2:")
print(ds_avg_anom)

# Save to NetCDF
avg_output_file = model_dir / f"cmip6_{file_model}_temperature_anomalies_1961-1990baseline.nc"
ds_avg_anom.to_netcdf(avg_output_file)
print(f"\n✓ Saved averaged anomalies to: {avg_output_file}")


## 7. Clean Up Intermediate Files

Remove per-variable combined files, keeping only original downloaded files and the final averaged anomaly file.


In [ ]:
cleanup_intermediate_files(model_dir, file_model)

print("\n" + "="*60)
print("Final files in directory:")
print("="*60)
for f in sorted(model_dir.glob("*.nc")):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name} ({size_mb:.1f} MB)")
print("="*60)
